# Stage 2 — Image acquisition and the coverage report

Filters the iNaturalist open-data release to research-grade Danish observations with
openly licensed photos, then reports how many taxa survive each minimum-observation
threshold.

**This stage stops and asks.** The taxon count sets the model's output dimension, the APK
size, and every accuracy figure downstream. Read the report, pick a threshold, re-run
with `--commit`.

**No GPU needed.** The archive is large; prefer a high-RAM runtime.


In [ ]:
#@title Mount Drive and install the package
from google.colab import drive
drive.mount('/content/drive')

CACHE = '/content/drive/MyDrive/life-list/cache'  #@param {type:"string"}
REPO  = '/content/life-list'                       #@param {type:"string"}

import os
os.makedirs(CACHE, exist_ok=True)

if not os.path.exists(REPO):
    !git clone https://github.com/StefanWiswedel/Life-List.git {REPO}
else:
    !cd {REPO} && git pull --ff-only

!pip -q install -e {REPO}/training
print('cache:', CACHE)


## Fetch the archive

One archive, not six separate files — the original brief had this wrong. Roughly 30-60
minutes, cached to Drive so it only happens once.


In [ ]:
ARCHIVE = f'{CACHE}/inaturalist-open-data-latest.tar.gz'

import os
if not os.path.exists(ARCHIVE):
    !wget -q --show-progress -O {ARCHIVE} \
        https://inaturalist-open-data.s3.amazonaws.com/metadata/inaturalist-open-data-latest.tar.gz
else:
    print('already downloaded')

!ls -lh {ARCHIVE}


## Confirm the schema before trusting the filter

The public README does not document the columns. A wrong guess would not raise — it would
filter to zero rows and read as "Denmark has no research-grade observations", which looks
like a fact about the data rather than a bug in the code.

Look at these headers before believing any number below.


In [ ]:
import tarfile

with tarfile.open(ARCHIVE, 'r:*') as tar:
    for member in tar.getmembers():
        if not member.name.endswith('.csv'):
            continue
        handle = tar.extractfile(member)
        if handle is None:
            continue
        print(f'{member.name}:')
        print(f'  {handle.readline().decode().strip()}\n')


## The report

In [ ]:
!lifelist-images \
    --archive {ARCHIVE} \
    --thresholds 50 80 120 200 \
    --min-observations 80 \
    --cache-dir {CACHE} \
    --verbose


## Decide, then commit

Set `--min-observations` to whatever the report justifies and re-run with `--commit`.

The tradeoff: a lower threshold covers more of the Danish biota but leaves each class
with less data, and thin classes are where a model produces confident nonsense. The
rollup exists to catch that, but it is better not to manufacture it in the first place.


In [ ]:
!lifelist-images \
    --archive {ARCHIVE} \
    --min-observations 80 \
    --max-photos-per-taxon 500 \
    --cache-dir {CACHE} \
    --commit \
    --verbose
